# TimeSeriesPolars — Polars Backend

The `TimeSeriesPolars` class is a Polars-backed container for time series data. It keeps everything in a `polars.DataFrame` — making it ideal for high-performance in-memory processing and database integration.

Key characteristics:

- **Four data shapes**: SIMPLE, VERSIONED, CORRECTED, AUDIT — matching the read patterns of a bi-temporal database
- **Polars-native**: data is stored as a `pl.DataFrame`; zero-copy interop with Arrow-based systems
- **Rich metadata**: name, unit, data type, geographic location, timezone, frequency, and arbitrary labels
- **Pandas interop**: `from_pandas()` / `to_pandas()` with automatic index handling per shape
- **Unit conversion**: `convert_unit()` for value scaling via pint

## Installation

```bash
pip install timedatamodel[pandas]
# or with all extras
pip install timedatamodel[all]
```

In [1]:
import pandas as pd
import polars as pl

import timedatamodel as tdm
from timedatamodel.timeseries_polars import TimeSeriesPolars, DataShape
from timedatamodel.enums import DataType, Frequency, TimeSeriesType
from timedatamodel.location import GeoLocation

## Quick start — SIMPLE shape

The most common shape is **SIMPLE**: one observation per `valid_time`. Use `TimeSeriesPolars.from_pandas()` to create one from a DataFrame.

In [2]:
df_simple = pd.DataFrame({
    "valid_time": pd.date_range("2024-01-15", periods=24, freq="1h", tz="UTC"),
    "value": [
        120.0, 115.0, 108.0, 105.0, 102.0, 100.0,
        110.0, 135.0, 160.0, 175.0, 180.0, 178.0,
        172.0, 170.0, 168.0, 165.0, 175.0, 190.0,
        200.0, 195.0, 180.0, 165.0, 145.0, 130.0,
    ],
})

ts = TimeSeriesPolars.from_pandas(
    df_simple,
    name="wind_power",
    unit="MW",
    frequency=Frequency.PT1H,
    data_type=DataType.OBSERVATION,
    location=GeoLocation(latitude=57.7, longitude=11.97),
    timezone="Europe/Stockholm",
    description="Hourly wind power output from farm Alpha",
)
ts

Name,wind_power
Shape,SIMPLE
Rows,24
Frequency,PT1H
Timezone,Europe/Stockholm
Unit,MW
Data type,OBSERVATION
Location,"57.7°N, 11.97°E"
Description,Hourly wind power output from farm Alpha
valid_time,wind_power
2024-01-15 00:00,120.0


The `valid_time` column can also live in the DataFrame index — `from_pandas()` handles both layouts:

In [3]:
df_indexed = df_simple.set_index("valid_time")
ts_from_index = TimeSeriesPolars.from_pandas(df_indexed, name="wind_power", unit="MW")

print(f"Shape: {ts_from_index.shape}")
print(f"Rows:  {ts_from_index.num_rows}")

Shape: DataShape.SIMPLE
Rows:  24


## Key properties

| Property | Description |
|---|---|
| `shape` | `DataShape` enum — which temporal columns are present |
| `num_rows` | Number of rows in the Polars DataFrame |
| `columns` | List of column names |
| `df` | The underlying `polars.DataFrame` (read-only by convention) |
| `has_missing` | `True` if the `value` column contains any nulls |

In [4]:
print(f"shape:       {ts.shape}")
print(f"num_rows:    {ts.num_rows}")
print(f"columns:     {ts.columns}")
print(f"len():       {len(ts)}")
print(f"has_missing: {ts.has_missing}")
print()
print("Underlying Polars schema:")
print(ts.df.schema)

shape:       DataShape.SIMPLE
num_rows:    24
columns:     ['valid_time', 'value']
len():       24
has_missing: False

Underlying Polars schema:
Schema({'valid_time': Datetime(time_unit='us', time_zone='UTC'), 'value': Float64})


## Creating directly from a Polars DataFrame

`TimeSeriesPolars.from_polars()` is the native constructor — use it when data is already in a `pl.DataFrame` (e.g. fetched from a database via `connectorx` or `adbc`). All timestamp columns must use `pl.Datetime("us", time_zone="UTC")`.

In [5]:
pl_df = pl.DataFrame({
    "valid_time": pd.date_range("2024-01-15", periods=6, freq="1h", tz="UTC"),
    "value": [100.0, 105.0, None, 108.0, 103.0, 98.0],
}).with_columns(
    pl.col("valid_time").cast(pl.Datetime("us", time_zone="UTC"))
)

ts_from_polars = TimeSeriesPolars.from_polars(
    pl_df,
    name="load",
    unit="MW",
    frequency=Frequency.PT1H,
    data_type=DataType.OBSERVATION,
)
print(f"has_missing: {ts_from_polars.has_missing}")
ts_from_polars

has_missing: True


Name,load
Shape,SIMPLE
Rows,6
Frequency,PT1H
Timezone,UTC
Unit,MW
Data type,OBSERVATION
valid_time,load
2024-01-15 00:00,100.0
2024-01-15 01:00,105.0
2024-01-15 02:00,NaN


## The four data shapes

`DataShape` is inferred automatically from the column names in the Arrow table.

| Shape | Temporal columns | Use case |
|---|---|---|
| **SIMPLE** | `valid_time` | Flat, immutable observations |
| **VERSIONED** | `knowledge_time`, `valid_time` | Forecast revision history |
| **CORRECTED** | `valid_time`, `change_time` | Corrected meter readings (no forecast) |
| **AUDIT** | `knowledge_time`, `change_time`, `valid_time` | Full bi-temporal log |

### VERSIONED shape

A `VERSIONED` series stores the full forecast revision history — one row per `(knowledge_time, valid_time)` pair. This is the standard output of a forecast-issuing system where new runs update future values.

In [6]:
# Two forecast runs (knowledge_times), each covering 6 hours ahead
rows = []
for kt_offset, values in [
    (0, [100.0, 105.0, 110.0, 108.0, 103.0, 98.0]),
    (6, [102.0, 107.0, 112.0, 111.0, 106.0, 101.0]),
]:
    kt = pd.Timestamp("2024-01-15 00:00", tz="UTC") + pd.Timedelta(hours=kt_offset)
    for h, v in enumerate(values):
        rows.append({
            "knowledge_time": kt,
            "valid_time": pd.Timestamp("2024-01-15 00:00", tz="UTC") + pd.Timedelta(hours=h),
            "value": v,
        })

df_versioned = pd.DataFrame(rows)

ts_v = TimeSeriesPolars.from_pandas(
    df_versioned,
    name="wind_power_forecast",
    unit="MW",
    frequency=Frequency.PT1H,
    data_type=DataType.FORECAST,
    timeseries_type=TimeSeriesType.OVERLAPPING,
)
ts_v

TimeSeriesPolars
┌───────────────────────────────────────────────────────────┐
│  Name:             wind_power_forecast                    │
│  Shape:            VERSIONED                              │
│  Rows:             12                                     │
│  Frequency:        PT1H                                   │
│  Timezone:         UTC                                    │
│  Unit:             MW                                     │
│  Data type:        FORECAST                               │
│  Timeseries type:  OVERLAPPING                            │
├───────────────────────────────────────────────────────────┤
│                                      wind_power_forecast  │
│  2024-01-15 00:00, 2024-01-15 00:00                100.0  │
│  2024-01-15 00:00, 2024-01-15 01:00                105.0  │
│  2024-01-15 00:00, 2024-01-15 02:00                110.0  │
│  ...                                                 ...  │
│  2024-01-15 06:00, 2024-01-15 03:00                111.0  │
│  2024-01-15 06:00, 2024-01-15 04:00                106.0  │
│  2024-01-15 06:00, 2024-01-15 05:00                101.0  │
└───────────────────────────────────────────────────────────┘

### CORRECTED shape

A `CORRECTED` series captures historical corrections to flat (non-forecast) data — for example, when sensor readings are recalibrated. Each correction adds a `change_time` column recording when the edit was made. CORRECTED shapes can only be created via `from_polars()` (read from a database), not `from_pandas()`.

In [7]:
corrected_df = pl.DataFrame({
    "valid_time":  [
        "2024-01-15 00:00:00",
        "2024-01-15 01:00:00",
        "2024-01-15 01:00:00",  # second row is corrected
    ],
    "change_time": [
        "2024-01-20 09:00:00",
        "2024-01-20 09:00:00",
        "2024-01-22 14:30:00",  # later correction
    ],
    "value": [120.0, 115.0, 116.5],  # 116.5 corrects 115.0
}).with_columns([
    pl.col("valid_time").str.to_datetime().dt.replace_time_zone("UTC"),
    pl.col("change_time").str.to_datetime().dt.replace_time_zone("UTC"),
])

ts_c = TimeSeriesPolars.from_polars(
    corrected_df,
    name="meter_reading",
    unit="MWh",
    data_type=DataType.OBSERVATION,
)
ts_c

TimeSeriesPolars
┌─────────────────────────────────────────────────────┐
│  Name:             meter_reading                    │
│  Shape:            CORRECTED                        │
│  Rows:             3                                │
│  Timezone:         UTC                              │
│  Unit:             MWh                              │
│  Data type:        OBSERVATION                      │
├─────────────────────────────────────────────────────┤
│                                      meter_reading  │
│  2024-01-15 00:00, 2024-01-20 09:00          120.0  │
│  2024-01-15 01:00, 2024-01-20 09:00          115.0  │
│  2024-01-15 01:00, 2024-01-22 14:30          116.5  │
└─────────────────────────────────────────────────────┘

## Data access — head, tail, has_missing

`head()` and `tail()` return a new `TimeSeriesPolars` with the first or last *n* rows, preserving all metadata. `has_missing` is a quick boolean check for null values in the `value` column.

In [8]:
print(f"has_missing: {ts.has_missing}")
print()
print("head(3):")
print(ts.head(3).df)
print()
print("tail(3):")
print(ts.tail(3).df)

has_missing: False

head(3):
shape: (3, 2)
┌─────────────────────────┬───────┐
│ valid_time              ┆ value │
│ ---                     ┆ ---   │
│ datetime[μs, UTC]       ┆ f64   │
╞═════════════════════════╪═══════╡
│ 2024-01-15 00:00:00 UTC ┆ 120.0 │
│ 2024-01-15 01:00:00 UTC ┆ 115.0 │
│ 2024-01-15 02:00:00 UTC ┆ 108.0 │
└─────────────────────────┴───────┘

tail(3):
shape: (3, 2)
┌─────────────────────────┬───────┐
│ valid_time              ┆ value │
│ ---                     ┆ ---   │
│ datetime[μs, UTC]       ┆ f64   │
╞═════════════════════════╪═══════╡
│ 2024-01-15 21:00:00 UTC ┆ 165.0 │
│ 2024-01-15 22:00:00 UTC ┆ 145.0 │
│ 2024-01-15 23:00:00 UTC ┆ 130.0 │
└─────────────────────────┴───────┘


## Coverage bar

`coverage_bar()` returns a `CoverageBar` object that renders as an SVG in Jupyter — each bin is green when at least one value in that bin is present, and gray when all values are null.

In [9]:
# ts has no missing values — fully filled bar
ts.coverage_bar()

wind_power  ████████████████████████
            2024-01-15 00:00  2024-01-15 23:00

In [10]:
# ts_from_polars has one null — the gap shows up as a gray bin
ts_from_polars.coverage_bar()

load  ██░███
      2024-01-15 00:00  2024-01-15 05:00

## Unit conversion

`convert_unit()` returns a new `TimeSeriesPolars` with values scaled to the target unit using [pint](https://pint.readthedocs.io/). The `unit` metadata field is updated automatically — all other metadata is preserved.

In [11]:
ts_mw = TimeSeriesPolars.from_pandas(
    df_simple,
    name="wind_power",
    unit="MW",
    frequency=Frequency.PT1H,
)

ts_gw = ts_mw.convert_unit("GW")

print(f"Original:  unit={ts_mw.unit!r},  first value = {ts_mw.df['value'][0]:.1f}")
print(f"Converted: unit={ts_gw.unit!r}, first value = {ts_gw.df['value'][0]:.4f}")

Original:  unit='MW',  first value = 120.0
Converted: unit='GW', first value = 0.1200


## Converting to pandas

`to_pandas()` restores the conventional index used by the existing read API. The index structure depends on the shape:

| Shape | Index |
|---|---|
| SIMPLE | `valid_time` |
| VERSIONED | `(knowledge_time, valid_time)` MultiIndex |
| CORRECTED | `(valid_time, change_time)` MultiIndex |
| AUDIT | `(knowledge_time, change_time, valid_time)` MultiIndex |

In [12]:
df_out = ts.to_pandas()
print(f"Index name: {df_out.index.name}")
df_out.head()

Index name: valid_time


,value
valid_time,
2024-01-15 00:00:00+00:00,120.0
2024-01-15 01:00:00+00:00,115.0
2024-01-15 02:00:00+00:00,108.0
2024-01-15 03:00:00+00:00,105.0
2024-01-15 04:00:00+00:00,102.0


In [13]:
df_v_out = ts_v.to_pandas()
print(f"Index names: {df_v_out.index.names}")
df_v_out.head(8)

Index names: ['knowledge_time', 'valid_time']


value
knowledge_time            valid_time                      
2024-01-15 00:00:00+00:00 2024-01-15 00:00:00+00:00  100.0
                          2024-01-15 01:00:00+00:00  105.0
                          2024-01-15 02:00:00+00:00  110.0
                          2024-01-15 03:00:00+00:00  108.0
                          2024-01-15 04:00:00+00:00  103.0
                          2024-01-15 05:00:00+00:00   98.0
2024-01-15 06:00:00+00:00 2024-01-15 00:00:00+00:00  102.0
                          2024-01-15 01:00:00+00:00  107.0

## Other conversion methods

`to_polars()`, `to_list()`, `to_numpy()`, and `to_pyarrow()` all return the **full table** — all columns including timestamps.

| Method | Returns | Extra dependency |
|---|---|---|
| `to_polars()` | `pl.DataFrame` | none |
| `to_list()` | `dict[str, list]` — column-oriented | none |
| `to_numpy()` | structured `np.ndarray` | `numpy` |
| `to_pyarrow()` | `pa.Table` | `pyarrow` |

In [ ]:
# No extra dependencies needed
print("to_polars():", type(ts.to_polars()))
print()
cols = ts.to_list()
print("to_list() keys:", list(cols.keys()))
print("to_list() first value:", cols["value"][0])
print()

# Requires numpy
import numpy as np
arr = ts.to_numpy()
print("to_numpy() dtype:", arr.dtype)
print("to_numpy() first row:", arr[0])
print()

# Requires pyarrow
import pyarrow as pa
tbl = ts.to_pyarrow()
print("to_pyarrow() schema:", tbl.schema)

## Metadata

`metadata_dict()` returns all metadata fields as a plain dict — useful for serialisation or logging.

In [15]:
import json
print(json.dumps(ts.metadata_dict(), indent=2, default=str))

{
  "name": "wind_power",
  "description": "Hourly wind power output from farm Alpha",
  "unit": "MW",
  "labels": {},
  "timezone": "Europe/Stockholm",
  "frequency": "PT1H",
  "data_type": "OBSERVATION",
  "location": {
    "longitude": 11.97,
    "latitude": 57.7
  },
  "timeseries_type": "FLAT",
  "shape": "SIMPLE",
  "num_rows": 24
}


## Labels

Arbitrary key-value labels let you tag series for filtering and provenance. They appear in the repr when set.

In [16]:
ts_labelled = TimeSeriesPolars.from_pandas(
    df_simple,
    name="wind_power",
    unit="MW",
    frequency=Frequency.PT1H,
    labels={"site": "Alpha", "turbine_model": "V150-4.5", "country": "SE"},
)
ts_labelled

Name,wind_power
Shape,SIMPLE
Rows,24
Frequency,PT1H
Timezone,UTC
Unit,MW
Labels,"{'site': 'Alpha', 'turbine_model': 'V150-4.5', 'country': 'SE'}"
valid_time,wind_power
2024-01-15 00:00,120.0
2024-01-15 01:00,115.0
2024-01-15 02:00,108.0


## Validating for database insert

`validate_for_insert()` confirms that the series can be written back to the database and returns `(pl.DataFrame, DataShape)`. Only **SIMPLE** and **VERSIONED** shapes are insertable — **CORRECTED** and **AUDIT** shapes are read-only query results.

In [17]:
df_insert, shape = ts.validate_for_insert()
print(f"Shape:   {shape}")
print(f"Columns: {df_insert.columns}")
print(f"Rows:    {df_insert.height}")
print(f"Ready to insert: ✓")

Shape:   DataShape.SIMPLE
Columns: ['valid_time', 'value']
Rows:    24
Ready to insert: ✓


In [18]:
# CORRECTED shape raises ValueError — it is a read-only audit result
try:
    ts_c.validate_for_insert()
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: TimeSeriesPolars with shape CORRECTED cannot be inserted. Only SIMPLE and VERSIONED shapes are supported for insert.


## DataShape enum

`DataShape` is exported from `timedatamodel` and can be used to branch on the shape of an incoming series:

In [19]:
for series in [ts, ts_v, ts_c]:
    match series.shape:
        case DataShape.SIMPLE:
            print(f"{series.name!r:30s} → flat observation, {series.num_rows} rows")
        case DataShape.VERSIONED:
            print(f"{series.name!r:30s} → forecast history, {series.num_rows} rows")
        case DataShape.CORRECTED:
            print(f"{series.name!r:30s} → corrected readings, {series.num_rows} rows")
        case DataShape.AUDIT:
            print(f"{series.name!r:30s} → full audit log, {series.num_rows} rows")

'wind_power'                   → flat observation, 24 rows
'wind_power_forecast'          → forecast history, 12 rows
'meter_reading'                → corrected readings, 3 rows


## Summary

In this notebook you learned how to:

- Create a `TimeSeriesPolars` from a pandas DataFrame (`from_pandas`) or a Polars DataFrame (`from_polars`)
- Understand the four **data shapes** (SIMPLE, VERSIONED, CORRECTED, AUDIT) and when each is used
- Inspect key properties: `shape`, `num_rows`, `columns`, `df`, `has_missing`
- Access first/last rows with `head()` and `tail()`
- Visualise value coverage with `coverage_bar()` — SVG in Jupyter, Unicode blocks in terminal
- Convert values to a different unit with `convert_unit()` (pint-powered)
- Convert to different formats with `to_pandas()`, `to_polars()`, `to_list()`, `to_numpy()`, and `to_pyarrow()`
- Attach metadata: name, unit, data type, location, timezone, frequency, and arbitrary labels
- Validate a series for database insert with `validate_for_insert()`
- Branch on shape using `DataShape` and `match`